# Metrics and Tracing

This notebook covers:

1. How **metrics, logs, and traces** differ — and what question each one answers
2. `prometheus_client` primitives: `Counter`, `Histogram`, `Gauge`
3. The `/metrics` endpoint Prometheus scrapes
4. Auto-instrumentation with `prometheus-fastapi-instrumentator` (request count, latency, in-progress)
5. OpenTelemetry: spans, the console exporter, the OTLP migration
6. Joining traces and logs: a `trace_id` field on every log line

**Scope**: FastAPI + `prometheus_client` + `prometheus-fastapi-instrumentator` + `opentelemetry-api`/`opentelemetry-sdk` + `structlog` (carried over from 6.2).

## 1. Metrics, Logs, and Traces — Three Pillars

All three describe the same running system from different angles. Different angle → different question answered → different storage shape.

| Pillar  | Shape                                  | Best at                                              | Worst at                            |
|---------|----------------------------------------|------------------------------------------------------|-------------------------------------|
| **Metrics** | Numbers aggregated over time         | "How many requests per second?" "p95 latency?"       | "Why is that specific request slow?" |
| **Logs**    | Discrete structured events           | "What happened during request `req_abc`?"            | Counting/aggregating millions of them |
| **Traces**  | Tree of operations across services   | "Where did the time go in this request?"             | Pre-computed numerical summaries     |

Use them together:

- An **alert** fires off a *metric* ("p95 latency on `/assets` exceeded 500ms for 5 minutes").
- You **jump to logs** filtered by the offending route and time window — that's why we keep `request_id` and `trace_id` on every line (6.2).
- You **pull a trace** for a slow request to see *which downstream call* burned the time.

This notebook wires up metrics (sections 2–4) and tracing (5–6) and threads the trace ID through structlog so the alert → logs → trace path is one click.

## 2. `prometheus_client` Basics

Prometheus is *pull-based*: your app exposes a `/metrics` endpoint, the Prometheus server scrapes it every N seconds, stores time series, alerts on PromQL queries. The Python library (`prometheus_client`) gives you four primitives:

- **`Counter`** — monotonically increasing. "Number of requests." Reset only by process restart. You ask Prometheus for the rate (`rate(requests_total[5m])`).
- **`Gauge`** — goes up and down. "In-flight requests," "queue depth," "connection pool size." Reset by direct assignment.
- **`Histogram`** — distribution with predefined buckets. "Request latency." Lets you compute percentiles (p50, p95, p99) server-side via `histogram_quantile`.
- **`Summary`** — like Histogram but with client-side quantile estimation. Almost always prefer Histogram in microservices (aggregable across instances).

In [ ]:
from prometheus_client import Counter, Gauge, Histogram, CollectorRegistry, generate_latest, CONTENT_TYPE_LATEST

# Use a dedicated registry to keep the demo isolated from the global one.
# Production apps usually use the default global registry — no `registry=` arg needed.
registry = CollectorRegistry()

asset_creates = Counter(
    "asset_creates_total",                       # name: foo_total for counters
    "Number of assets created",                  # help text
    ["asset_class"],                             # labels — one time series per unique label value
    registry=registry,
)

in_flight = Gauge(
    "http_requests_in_flight",
    "Number of in-flight HTTP requests",
    registry=registry,
)

request_latency = Histogram(
    "http_request_duration_seconds",
    "HTTP request latency",
    ["method", "route"],
    # Buckets in seconds: tune for your latency budget. These are good defaults for web APIs.
    buckets=(0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0),
    registry=registry,
)

# Drive each one to generate sample data.
asset_creates.labels(asset_class="equity").inc()
asset_creates.labels(asset_class="equity").inc()
asset_creates.labels(asset_class="bond").inc()

in_flight.inc(3)
in_flight.dec()  # one request finished

with request_latency.labels(method="GET", route="/assets/{ticker}").time():
    # The context manager records the wall-clock duration into the histogram.
    import time; time.sleep(0.02)

# generate_latest renders the registry as the Prometheus text exposition format.
print(generate_latest(registry).decode()[:1500])

What that output is showing — quick tour:

- **`# HELP` / `# TYPE`** lines are scraped as metric metadata.
- **`asset_creates_total{asset_class="equity"} 2`** — one *time series* per label combination. Two creates of equities = 2.
- **`http_requests_in_flight 2`** — the gauge's current value (set 3, decremented 1).
- **`http_request_duration_seconds_bucket{...,le="0.025"} 1`** — histogram buckets. The `le="+Inf"` bucket always exists; together they let PromQL compute quantiles.

Three label-design rules that save your future self:

- **Bounded cardinality.** *Never* use a label whose value can be unique per request (`user_id`, `request_id`, `trace_id`). Each unique value creates a new time series. A million users → a million series → an unhappy Prometheus.
- **Routes, not paths.** Label by `/assets/{ticker}` (the route pattern), not `/assets/AAPL` (the concrete URL). Same reason.
- **Status code class, sometimes status code.** `2xx`/`4xx`/`5xx` is fine; full status codes (`200`, `201`, `404`, `429`, `500`) is also fine — both are bounded.

High-cardinality data belongs in **logs** (6.2) or **traces** (section 5), not in metric labels.

## 3. The `/metrics` Endpoint

Prometheus expects an HTTP endpoint that responds with the text format above. Wire it up as a FastAPI route — one line, no middleware, no auth (typically):

In [ ]:
from fastapi import FastAPI, Response
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/metrics", include_in_schema=False)  # exclude from /docs — it's an ops endpoint
def metrics():
    return Response(content=generate_latest(registry), media_type=CONTENT_TYPE_LATEST)

@app.get("/assets/{ticker}")
def get_asset(ticker: str):
    asset_creates.labels(asset_class="equity").inc()  # contrived, just to bump the counter
    return {"ticker": ticker}

client = TestClient(app)

# Burst some traffic, then scrape.
for t in ["AAPL", "MSFT", "NVDA"]:
    client.get(f"/assets/{t}")

scrape = client.get("/metrics")
print("status      :", scrape.status_code)
print("content-type:", scrape.headers["content-type"])
print("\n--- prometheus text format ---")
print(scrape.text[:600])

Three properties of the `/metrics` route to internalize:

- **`include_in_schema=False`** keeps the ops endpoint out of `/docs`. Nobody making business-API calls cares about it.
- **Cheap to scrape.** `generate_latest()` walks the in-memory registry. No DB calls, no I/O — milliseconds even with thousands of series.
- **No auth (usually).** Prometheus on the same network scrapes it. If the endpoint is public, gate it with the rate limiter from 5.3 or a private port; never expose it directly to the internet (it leaks operational info like route names and request counts).

## 4. Auto-Instrumentation with `prometheus-fastapi-instrumentator`

Hand-rolling Counters and Histograms for every route is tedious and gets it wrong subtly (forgetting to bump on error paths, double-counting on retries). `prometheus-fastapi-instrumentator` is the FastAPI-native equivalent of stdlib's auto-instrumentation: it hooks the middleware stack and exposes the standard four metrics (request count, latency, in-progress, response size) automatically. You can still add custom counters on top.

In [ ]:
from prometheus_fastapi_instrumentator import Instrumentator

app2 = FastAPI()

# .instrument(app) attaches the middleware; .expose(app) adds /metrics for you.
Instrumentator().instrument(app2).expose(app2, include_in_schema=False)

@app2.get("/assets/{ticker}")
def get_asset(ticker: str):
    return {"ticker": ticker}

@app2.post("/assets", status_code=201)
def create_asset():
    return {"created": True}

@app2.get("/missing")
def missing():
    from fastapi import HTTPException
    raise HTTPException(status_code=404, detail="not here")

c2 = TestClient(app2)

# Drive a few requests in: 2xx, 404, mixed methods.
for _ in range(3): c2.get("/assets/AAPL")
for _ in range(2): c2.post("/assets")
for _ in range(1): c2.get("/missing")

scrape = c2.get("/metrics").text

# Just print the lines that are most representative — full output is hundreds of lines.
interesting = [
    line for line in scrape.splitlines()
    if line.startswith("http_requests_total{") and "handler=" in line
    and ("/assets" in line or "/missing" in line)
]
print("\n".join(interesting[:8]))

Each line is one time series tagged by method, route handler, and status — exactly the labels you'd hand-roll if you cared, but the instrumentator did it for you. Notice **the label is `handler="/assets/{ticker}"`**, not `/assets/AAPL` — the route pattern, not the concrete URL. That's the cardinality discipline from section 2, applied automatically.

Two production patterns:

- **Add custom counters on top.** Business metrics (`asset_creates_total`, `order_value_usd_sum`) live alongside the HTTP metrics; the instrumentator doesn't interfere with the registry.
- **Exclude `/health` and `/metrics`.** Both are polled constantly by the LB and the scraper, respectively. They dominate the request-count metric otherwise. `Instrumentator(excluded_handlers=["/health", "/metrics"])` filters them.

## 5. OpenTelemetry Tracing

Where metrics aggregate ("how many requests?") and logs record ("what happened?"), **traces show structure** — a single request becomes a tree of operations ("span" in OTel parlance): the route handler, the auth check, the DB query, the upstream API call. Each span has a duration and parent/child links. The whole tree shares a single `trace_id`.

**OpenTelemetry (OTel)** is the vendor-neutral standard. The Python SDK splits into:

- **API** (`opentelemetry-api`) — what your code calls (`tracer.start_as_current_span(...)`). Stable.
- **SDK** (`opentelemetry-sdk`) — the implementation that produces spans. Configured at startup.
- **Exporters** — where spans go: `ConsoleSpanExporter` (stdout, for dev), `OTLPSpanExporter` (production: send to Jaeger / Tempo / Honeycomb / Datadog over OTLP/gRPC).

This notebook uses the console exporter so you can read spans directly in the output. In production you'd swap one line:

```python
# from: dev/tutorial
exporter = ConsoleSpanExporter()
# to:   prod
exporter = OTLPSpanExporter(endpoint="http://otel-collector:4317")
```

Same code calls `tracer.start_as_current_span` — the exporter is the only thing that changes.

In [ ]:
import io
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Capture span output so we can inspect it instead of dumping to real stdout.
span_buffer = io.StringIO()

# Set up a tracer provider exactly once. In a real app this lives in observability.py.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter(out=span_buffer)))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("portfolio-api")  # logical service name

# Hand-rolled instrumentation: open a span around any operation you want timed in the trace.
with tracer.start_as_current_span("get_asset") as outer:
    outer.set_attribute("ticker", "AAPL")
    with tracer.start_as_current_span("db_query") as inner:
        inner.set_attribute("db.statement", "SELECT * FROM assets WHERE ticker = 'AAPL'")
        # ... pretend DB call ...
    with tracer.start_as_current_span("price_calc") as inner:
        inner.set_attribute("algo", "vwap")
        # ... pretend computation ...

# Spans are emitted (printed by the console exporter) when they end. Read them back here.
raw = span_buffer.getvalue()
print(raw[:1200])

Three things to look for in the output:

- **All three spans share the same `trace_id`.** That's the correlation ID that ties everything together. (You'll see it as a hex string — `0x...`.)
- **Parent-child structure** is encoded in `parent_id`: `db_query`'s `parent_id` matches `get_asset`'s `span_id`. A trace UI renders this as a flame graph.
- **Attributes** are key/value tags on a span. *This* is where high-cardinality data goes — `ticker="AAPL"`, `db.statement="..."`, `user_id=12345`. Unlike metric labels, trace attributes don't create new time series.

For real FastAPI apps you almost never hand-roll spans. **Auto-instrumentation** wraps the HTTP layer:

```python
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
FastAPIInstrumentor.instrument_app(app)   # one line, one span per request
```

Every request gets a parent span automatically; hand-rolled spans nest under it. The same package family provides instrumentation for httpx, sqlalchemy, redis, etc. — turn them on once at startup and the trace tree forms itself.

## 6. Joining Traces and Logs: `trace_id` on Every Log Line

The point of having all three pillars is using them together. The bridge is the **trace ID** — present in the trace UI *and* attached to every log line emitted during a request. One click in Tempo / Jaeger jumps to the matching logs in Loki / Datadog.

Wiring it in is one structlog processor that reads the current span and injects `trace_id` / `span_id` into the event dict:

In [ ]:
import logging, structlog

def add_trace_context(logger, method_name, event_dict):
    """structlog processor: stamp current trace_id / span_id from OTel into the log line."""
    span = trace.get_current_span()
    ctx = span.get_span_context() if span else None
    if ctx and ctx.is_valid:
        event_dict["trace_id"] = f"{ctx.trace_id:032x}"
        event_dict["span_id"]  = f"{ctx.span_id:016x}"
    return event_dict

log_buffer = io.StringIO()
structlog.configure(
    processors=[
        structlog.contextvars.merge_contextvars,    # request_id from middleware (6.2)
        add_trace_context,                          # trace_id from current span
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso", utc=True),
        structlog.processors.JSONRenderer(sort_keys=True),
    ],
    wrapper_class=structlog.make_filtering_bound_logger(logging.INFO),
    logger_factory=structlog.PrintLoggerFactory(file=log_buffer),
    cache_logger_on_first_use=False,
)
log = structlog.get_logger()

# A log call OUTSIDE any span — no trace_id field, expected.
log.info("startup_complete")

# A log call INSIDE a span — gets trace_id and span_id automatically.
with tracer.start_as_current_span("create_asset") as span:
    span.set_attribute("ticker", "AAPL")
    log.info("asset_created", ticker="AAPL", price=190)
    log.info("price_published", channel="sse")

print(log_buffer.getvalue().rstrip())

Notice the difference between the three lines:

- `startup_complete` has no `trace_id` — emitted outside any span. That's correct.
- `asset_created` and `price_published` *both* carry the same `trace_id` and `span_id` — they were emitted inside `create_asset`.

That's the join key for the alert → logs → trace path: alert fires → quote the trace_id from the trace UI → grep `"trace_id": "..."` in Loki → every log line for that request, in order.

Three operational patterns worth lifting wholesale:

- **Add the processor once at startup**, like every other structlog processor. It runs on every log call automatically.
- **`trace_id` in metrics is still off-limits.** It's high-cardinality. The trace ID belongs in *exemplars* (a Prometheus feature) — a fixed-size sample of trace IDs attached to histogram buckets — not in label values.
- **W3C trace context (`traceparent`, `tracestate`) headers** flow through HTTP automatically when both sides run OTel. That's how a trace can span the frontend, your API, and a downstream service: every hop reads the parent span ID from the incoming `traceparent` and makes its own a child.

## Key Takeaways

- **Three pillars, three jobs.** Metrics answer *how much*; logs answer *what happened in this request*; traces answer *where the time went*. None of the three is optional in a production API.
- **Counters, gauges, histograms** are the bread-and-butter Prometheus types. Histograms for latency; counters for events; gauges for in-flight state. **Bounded label cardinality** is the rule that keeps Prometheus alive.
- **`/metrics`** is a one-line route returning `generate_latest(registry)`. Exclude from `/docs`; keep behind an ops network.
- **`prometheus-fastapi-instrumentator`** gives you the standard four HTTP metrics for free. Stack custom counters on top.
- **OpenTelemetry traces** structure each request as a span tree. Console exporter for dev, OTLP for production — your `tracer.start_as_current_span` calls don't change.
- **`FastAPIInstrumentor.instrument_app(app)`** opens a span per request automatically. Hand-rolled spans nest under it.
- **Join logs to traces** with a structlog processor that reads `trace.get_current_span()` and stamps `trace_id` on the event dict. The alert → logs → trace path becomes one click.
- **Capstone tie-in**: `observability.py` will host (a) `Instrumentator().instrument(app).expose(app)`, (b) the OTel `TracerProvider` setup with OTLP-from-settings, (c) the `add_trace_context` processor in the structlog chain configured in 6.2.

## Exercises

**1. Instrument a custom business metric.** Add `orders_placed_total{asset_class}` and `order_value_usd_sum` counters. Bump them from a `POST /orders` route. Confirm both show up in `/metrics` with the right label values after a few orders. Sketch in a markdown cell which dashboard query you'd write (`rate(orders_placed_total[5m])`, `rate(order_value_usd_sum[5m]) / rate(orders_placed_total[5m])` for average order size).

**2. Enable auto-instrumentation for FastAPI.** Add `FastAPIInstrumentor.instrument_app(app)` to a small FastAPI app, hit a route via TestClient, and inspect the emitted spans from the console exporter. Confirm there's one auto-span per request and that any `tracer.start_as_current_span(...)` you add inside the handler nests under it. Then add `RequestsInstrumentor().instrument()` to instrument an outbound `httpx.get(...)` and confirm the new span shows up as a child too.

**3. Histograms with exemplars.** Pick a route. Replace the auto-instrumented latency histogram with a hand-rolled `Histogram(..., enable_exemplars=True)` and attach the current `trace_id` as an exemplar on `observe`. Look at the `/metrics` output: lines will include an `# {trace_id="..."} <value> <timestamp>` exemplar suffix. Sketch in a markdown cell why this is the right way to bridge metrics → traces for a slow request without putting trace IDs in labels.